In [ ]:
### 验证是否使用GPU
!nvidia-smi

# loading packages

In [15]:
! pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 17.4 MB/s  0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [16]:
import math
import numpy as np
import pandas as pd
import os 
import csv
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

# Some utility functions
## Utility function for reproducibilty

In [17]:
def same_seed(seed):
    ### cudnn deterministric = True
    ### cudnn benchmark = False
    ### np.random.seed()
    ### torch.manual_seed()
    ### torch.cuda.manual_seed()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.seed_all(seed)

## Function for split training data

In [34]:
def train_valid_split(dataset,ratio,seed):
    valid_set_size = int(ratio * len(dataset))
    train_set_size = len(dataset) - valid_set_size
    train_set, valid_set = random_split(dataset,[train_set_size,valid_set_size],
                                          generator=torch.Generator().manual_seed(seed))
    return np.array(train_set), np.array(valid_set)

## Function for prediction

In [35]:
def pred(test_loader,model,device):
    model.eval()
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu()) ### 链式调用 每个方法都有返回值
    preds = torch.cat(preds,dim=0).numpy()
    return preds


# Dataset函数

In [36]:
class MyDataSet(Dataset):
    def __init__(self,x, y = None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)
    
    def __getitem__(self, index):
        if self.y is None:
            return self.x[index]
        else:
            return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)
        

# Neural Network Sructure

In [37]:
class My_model(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1),
        )
    def forward(self,x):
        out = self.layers(x)
        out = out.squeenze(1)
        return out

# Feature Selection

In [38]:
def select_feat(train_data,valid_data,test_data, select_all = True):
    y_train, y_valid = train_data[:,-1], valid_data[:,-1]
    raw_train_x, raw_valid_x, raw_test_x = train_data[:,:-1], valid_data[:,:-1], test_data
    if select_all:
        columns = list(range(raw_train_x.shape[1]))
        return raw_train_x[:,columns], raw_valid_x[:,columns], raw_test_x[:,columns],y_train,y_valid
    else:
        columns = [0,1,2,3]
        return raw_train_x[:,columns], raw_valid_x[:,columns], raw_test_x[:,columns],y_train,y_valid

# Training Loop

In [39]:
def trainer(train_loader, valid_loader, model, config, device):
    criterion = nn.MSELoss(reduction= 'mean')
    optimizer = torch.optim.SGD(model.parameters(),lr = config['learning_rare'], momentum= 0.9)
    writer = SummaryWriter() ### 初始化一个tensorboard from tensor.

    if not os.path.isdir('./models'):
        os.mkdir('./models')
    
    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train()
        loss_record = []
        train_pbar = tqdm(train_loader,position=0,leave=True)
        for x, y in train_pbar:
            torch.no_grad()
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backwards() ### 这里其实算的是这一批的loss
            optimizer.step()
            loss_record.append(loss.detach().item())

            # Display current epoch number and loss on tqdm progress bar.
            train_pbar.set_description(f'Epoch [{epoch+1}/{n_epochs}]')
            train_pbar.set_postfix({'loss': loss.detach().item()})
        mean_valid_loss = sum(loss_record)/len(loss_record)
        writer.add_scalar('Loss/train',mean_valid_loss,step)
        ### 经过一个epoch的训练
        model.eval()
        loss_record = []
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)
            loss_record.append(loss.detach().item()) ### 每一批数据计算一个
        mean_valid_loss = sum(loss_record)/len(loss_record)
        print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_valid_loss:.4f}, Valid loss: {mean_valid_loss:.4f}')
        writer.add_scalar('Loss/valid', mean_valid_loss, step)
        
        ### save the best model
        if mean_valid_loss < best_loss:
            torch.save(model.state_dict(),config['save_model'])
            print('Saving model with loss {:.3f}...'.format(best_loss))
            early_stop_count = 0
        else:
            early_stop_count += 1
        
        if early_stop_count >= config['early_stop']:
            print('\nModel is not improving, so we halt the training session.')
            return

In [40]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = {
    'seed': 5201314,      # Your seed number, you can pick your lucky number. :)
    'select_all': True,   # Whether to use all features.
    'valid_ratio': 0.2,   # validation_size = train_size * valid_ratio
    'n_epochs': 3000,     # Number of epochs.            
    'batch_size': 256, 
    'learning_rate': 1e-5,              
    'early_stop': 400,    # If model has not improved for this many consecutive epochs, stop training.     
    'save_path': './models/model.ckpt'  # Your model will be saved here.
}

## DataLoader

In [ ]:
same_seed(seed=config['seed'])
train_data, test_data  = pd.read_csv('./covid.train.csv',).values, pd.read_csv('./covid.test.csv').values
train_data, valid_data = train_valid_split(train_data, config['valid_ratio'], seed= config['seed'])
### 
print(f'train data的形状是: {train_data.shape}, valid data的形状是{valid_data.shape}, test data的形状是{test_data.shape}.')
### 
train_data, valid_data, test_data, y_train, y_valid = select_feat(train_data, valid_data, test_data)
###
print(f"特征数量为{train_data.shape[1]}.")

train_dataset, valid_dataset, test_dataset = MyDataSet(train_data,y_train),\
                                                MyDataSet(valid_data,y_valid),\
                                                MyDataSet(test_data)
train_loader = DataLoader(train_data,batch_size=config['batch_size'],shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=config['batch_size'],shuffle=True)
test_loader = DataLoader(test_data,batch_size=config['batch_size'],shuffle=False)




train data的形状是: (2160, 118), valid data的形状是(539, 118), test data的形状是(1078, 117).
特征数量为117.
